# 00 - Configuracion Inicial del Proyecto
## Pipeline ELT - Bioactividad de Medicamentos (PubChem)
### Grupo: D_GRUPO_4

---

### Objetivo de este notebook

Este notebook crea toda la **infraestructura base** necesaria para ejecutar el pipeline de datos.
Antes de poder ingestar, transformar o analizar datos, es imprescindible contar con:

- **Esquemas** (bases de datos logicas) para cada capa de la arquitectura medallon
- **Volumenes** en Unity Catalog para almacenar archivos crudos (capa Raw)

### Arquitectura Medallon

El pipeline sigue el patron de arquitectura **medallon** (medallion architecture),
un estandar de la industria para pipelines de datos en plataformas lakehouse:

| Capa | Esquema | Proposito |
|------|---------|----------|
| **Raw** | `pubchem_raw` | Archivos JSON crudos tal como llegan del API |
| **Bronce** | `pubchem_bronce` | Datos en formato Delta sin transformaciones de negocio |
| **Plata** | `pubchem_plata` | Datos limpios, estandarizados y enriquecidos |
| **Oro** | `pubchem_oro` | Modelo estrella optimizado para consultas analiticas |

### Requisitos previos

- Acceso a un workspace de Databricks con Unity Catalog habilitado
- Permisos para crear esquemas y volumenes en el catalogo activo

> **Importante:** Ejecutar este notebook **una sola vez** antes de correr el pipeline completo.

---
## 1. Deteccion del Catalogo Activo

En Unity Catalog, cada workspace tiene un **catalogo activo** por defecto.
En lugar de codificar el nombre del catalogo de forma rigida (hardcode),
lo detectamos dinamicamente para que el pipeline sea **portable** entre
diferentes entornos (desarrollo, staging, produccion).

In [ ]:
# Detectar el catalogo activo del workspace de Unity Catalog
# Esto permite que el pipeline funcione en cualquier entorno sin modificar codigo
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]

# Mostrar el catalogo detectado para verificacion visual
print(f"Catalogo activo detectado: {CATALOGO}")

---
## 2. Variables de Configuracion

Definimos todas las variables de configuracion como **constantes** al inicio del notebook.
Esto facilita el mantenimiento: si en el futuro se necesita cambiar un nombre de esquema
o volumen, solo hay que modificarlo en un unico lugar.

Seguimos la convencion de Python de usar **MAYUSCULAS** para constantes.

In [ ]:
# Nombres de los esquemas para cada capa de la arquitectura medallon
# Cada esquema agrupa las tablas que pertenecen a una misma etapa del pipeline
ESQUEMA_RAW = "pubchem_raw"         # Capa de datos crudos (archivos JSON)
ESQUEMA_BRONCE = "pubchem_bronce"   # Capa bronce (Delta Tables sin transformar)
ESQUEMA_PLATA = "pubchem_plata"     # Capa plata (datos limpios y estandarizados)
ESQUEMA_ORO = "pubchem_oro"         # Capa oro (modelo estrella para analitica)

# Nombre del volumen donde se almacenaran los archivos JSON crudos
# Los volumenes en Unity Catalog son almacenamiento gestionado para archivos no tabulares
NOMBRE_VOLUMEN = "bioactividad"

# Mostrar la configuracion definida para verificacion
print("Configuracion de esquemas:")
print(f"  Raw:    {ESQUEMA_RAW}")
print(f"  Bronce: {ESQUEMA_BRONCE}")
print(f"  Plata:  {ESQUEMA_PLATA}")
print(f"  Oro:    {ESQUEMA_ORO}")
print(f"  Volumen: {NOMBRE_VOLUMEN}")

---
## 3. Creacion de Esquemas

Creamos los cuatro esquemas necesarios dentro del catalogo activo.
Utilizamos `CREATE SCHEMA IF NOT EXISTS` para que la operacion sea **idempotente**:
se puede ejecutar multiples veces sin error ni efectos secundarios.

Cada esquema actua como un **namespace logico** que agrupa las tablas de su capa correspondiente.

In [ ]:
# Lista de todos los esquemas que necesita el pipeline
esquemas = [ESQUEMA_RAW, ESQUEMA_BRONCE, ESQUEMA_PLATA, ESQUEMA_ORO]

# Iterar sobre cada esquema y crearlo si no existe
for esquema in esquemas:
    # Ejecutar DDL de creacion con nombre completo: catalogo.esquema
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")
    print(f"  Esquema creado/verificado: {CATALOGO}.{esquema}")

print("\nTodos los esquemas fueron configurados exitosamente.")

---
## 4. Creacion del Volumen para Archivos Crudos (Capa Raw)

Un **volumen** en Unity Catalog es un recurso de almacenamiento gestionado
que permite guardar archivos no tabulares (JSON, CSV, Parquet, etc.)
con gobernanza y control de acceso integrados.

Creamos el volumen `bioactividad` dentro del esquema `pubchem_raw` para almacenar
los archivos JSON que se descargaran del API de PubChem en el notebook de ingesta.

La ruta resultante sera: `/Volumes/{CATALOGO}/pubchem_raw/bioactividad`

In [ ]:
# Crear el volumen dentro del esquema raw
# El volumen es idempotente: si ya existe, no se modifica
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA_RAW}.{NOMBRE_VOLUMEN}")

# Construir la ruta completa del volumen en formato DBFS
# Esta ruta se usara en el notebook de ingesta para guardar los archivos JSON
RUTA_VOLUMEN_RAW = f"/Volumes/{CATALOGO}/{ESQUEMA_RAW}/{NOMBRE_VOLUMEN}"

print(f"Volumen creado/verificado: {RUTA_VOLUMEN_RAW}")

---
## 5. Verificacion de la Infraestructura

Es una buena practica verificar que toda la infraestructura se creo correctamente
antes de proceder con el pipeline. Realizamos dos verificaciones:

1. **Esquemas**: Listamos todos los esquemas del catalogo que coincidan con el patron `pubchem_*`
2. **Volumen**: Verificamos que el volumen es accesible listando su contenido

In [ ]:
# Verificacion 1: Listar esquemas creados que coincidan con el patron pubchem_*
# Esto confirma que los 4 esquemas existen en el catalogo
resultado = spark.sql(f"SHOW SCHEMAS IN {CATALOGO} LIKE 'pubchem_*'")
resultado.show(truncate=False)

In [ ]:
# Verificacion 2: Comprobar accesibilidad del volumen
# En la primera ejecucion el volumen estara vacio, lo cual es normal
try:
    # Intentar listar archivos dentro del volumen
    archivos = dbutils.fs.ls(RUTA_VOLUMEN_RAW)
    print(f"Volumen {RUTA_VOLUMEN_RAW}: {len(archivos)} archivo(s) encontrado(s)")
except:
    # Si el volumen esta vacio, dbutils puede lanzar excepcion
    print("Volumen vacio (comportamiento normal en la primera ejecucion)")

print("\nConfiguracion inicial completada. El pipeline esta listo para ejecutarse.")